# Lab | BabyAGI with agent

**Change the planner objective below by changing the objective and the associated prompts and potential tools and agents - Wear your creativity and AI engineering hats
You can't get this wrong!**

You would need the OpenAI API KEY and the [SerpAPI KEY](https://serpapi.com/manage-api-keyhttps://serpapi.com/manage-api-key) to run this lab.


## BabyAGI with Tools

This notebook builds on top of [baby agi](baby_agi.html), but shows how you can swap out the execution chain. The previous execution chain was just an LLM which made stuff up. By swapping it out with an agent that has access to tools, we can hopefully get real reliable information

## Install and Import Required Modules

In [1]:
!pip install langchain langchain-community langchain-experimental langchain-openai langchain-classic

Exception ignored in: <function _releaseLock at 0x101926cb0>
Traceback (most recent call last):
  File "/opt/anaconda3/envs/ironhack_langchain/lib/python3.10/logging/__init__.py", line 228, in _releaseLock
    def _releaseLock():
KeyboardInterrupt: 


In [2]:
from typing import Optional

# Legacy chains/prompts → langchain-classic
from langchain_classic.chains import LLMChain
from langchain_classic.prompts import PromptTemplate

# BabyAGI still in langchain-experimental
from langchain_experimental.autonomous_agents import BabyAGI

# OpenAI integrations
from langchain_openai import OpenAI, OpenAIEmbeddings

## Connect to the Vector Store

Depending on what vectorstore you use, this step may look different.

In [ ]:
# # %pip install faiss-cpu > /dev/null
# # %pip install google-search-results > /dev/null
# from langchain.docstore import InMemoryDocstore
# from langchain_community.vectorstores import FAISS

In [3]:
!pip install faiss-cpu langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 22.0 MB/s  0:00:00 eta 0:00:01


In [4]:
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore  # Updated path
from langchain_community.vectorstores import FAISS

In [5]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')
SERPAPI_API_KEY = os.getenv('SERPAPI_API_KEY')

In [6]:
# Define your embedding model
from langchain_openai import OpenAIEmbeddings
embeddings_model = OpenAIEmbeddings()

# Initialize the vectorstore as empty
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

embedding_size = 1536
index = faiss.IndexFlatL2(embedding_size)
vectorstore = FAISS(
    embedding_function=embeddings_model,  # Full embeddings object, not .embed_query
    index=index,
    docstore=InMemoryDocstore({}),
    index_to_docstore_id={}
)

## Define the Chains

BabyAGI relies on three LLM chains:
- Task creation chain to select new tasks to add to the list
- Task prioritization chain to re-prioritize tasks
- Execution Chain to execute the tasks


NOTE: in this notebook, the Execution chain will now be an agent.

In [7]:
!pip install google-search-results

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for google-search-results: filename=google_search_results-2.4.2-py3-none-any.whl size=32093 sha256=ec93b7223f9921ea5076872f5a231940b577166508775dac54f23f06944cc882
  Stored in directory: /Users/omari/Library/Caches/pip/wheels/d3/b2/c3/03302d12bb44a2cdff3c9371f31b72c0c4e84b8d2285eeac53
Successfully built google-search-results


In [8]:
from langchain_classic.agents import AgentExecutor, Tool, ZeroShotAgent  # Legacy agents
from langchain_classic.chains import LLMChain  # Legacy chains
from langchain_classic.prompts import PromptTemplate  # Legacy prompts
from langchain_community.utilities import SerpAPIWrapper
from langchain_openai import OpenAI
from langchain_openai import ChatOpenAI

todo_prompt = PromptTemplate.from_template(
    "You are a planner who is an expert at coming up with a todo list for a given objective. Come up with a todo list for this objective: {objective}"
)
todo_chain = LLMChain(llm=ChatOpenAI(model="gpt-4o-mini", temperature=0), prompt=todo_prompt)
search = SerpAPIWrapper()
tools = [
    Tool(
        name="Search",
        func=search.run,
        description="useful for when you need to answer questions about current events",
    ),
    Tool(
        name="TODO",
        func=todo_chain.run,
        description="useful for when you need to come up with todo lists. Input: an objective to create a todo list for. Output: a todo list for that objective. Please be very clear what the objective is!",
    ),
]


prefix = """You are an AI who performs one task based on the following objective: {objective}. Take into account these previously completed tasks: {context}."""
suffix = """Question: {task}
{agent_scratchpad}"""
prompt = ZeroShotAgent.create_prompt(
    tools,
    prefix=prefix,
    suffix=suffix,
    input_variables=["objective", "task", "context", "agent_scratchpad"],
)

/var/folders/81/7h5x9_fd11n5xw4dczdmcph80000gn/T/ipykernel_63915/349246153.py:11: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 2.0.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  todo_chain = LLMChain(llm=ChatOpenAI(model="gpt-4o-mini", temperature=0), prompt=todo_prompt)


In [9]:

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
llm_chain = LLMChain(llm=llm, prompt=prompt)
tool_names = [tool.name for tool in tools]
agent = ZeroShotAgent(llm_chain=llm_chain, allowed_tools=tool_names)
agent_executor = AgentExecutor.from_agent_and_tools(
    agent=agent, tools=tools, verbose=True
)

/var/folders/81/7h5x9_fd11n5xw4dczdmcph80000gn/T/ipykernel_63915/1591959347.py:4: LangChainDeprecationWarning: Use `langchain.agents.create_agent` for new applications. It provides a more flexible agent factory with middleware support, structured output, and integration with LangGraph for persistence, streaming, and human-in-the-loop workflows. Migration guide: https://docs.langchain.com/oss/python/migrate/langchain-v1
  agent = ZeroShotAgent(llm_chain=llm_chain, allowed_tools=tool_names)


### Run the BabyAGI

Now it's time to create the BabyAGI controller and watch it try to accomplish your objective.

In [10]:
OBJECTIVE = "Write a weather report for SF today"

In [11]:
# Logging of LLMChains
verbose = False
# If None, will keep on going forever
max_iterations: Optional[int] = 3
baby_agi = BabyAGI.from_llm(
    llm=llm,
    vectorstore=vectorstore,
    task_execution_chain=agent_executor,
    verbose=verbose,
    max_iterations=max_iterations,
)

In [12]:
baby_agi({"objective": OBJECTIVE})

/var/folders/81/7h5x9_fd11n5xw4dczdmcph80000gn/T/ipykernel_63915/3867971396.py:1: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  baby_agi({"objective": OBJECTIVE})



*****TASK LIST*****

1: Make a todo list

*****NEXT TASK*****

1: Make a todo list


> Entering new AgentExecutor chain...
Question: What is the weather report for San Francisco today?  
Thought: I need to find the current weather conditions for San Francisco.  
Action: Search  
Action Input: "current weather San Francisco"  
Observation: {'type': 'weather_result', 'temperature': '63', 'unit': 'Fahrenheit', 'precipitation': '0%', 'humidity': '70%', 'wind': '17 mph', 'location': 'San Francisco, CA', 'date': 'Wednesday 2:00 PM', 'weather': 'Sunny'}
Thought:I have gathered the current weather information for San Francisco.  
Final Answer: The weather report for San Francisco today is sunny with a temperature of 63°F. There is no precipitation expected, humidity is at 70%, and winds are blowing at 17 mph.

> Finished chain.

*****TASK RESULT*****

The weather report for San Francisco today is sunny with a temperature of 63°F. There is no precipitation expected, humidity is at 70%, and win

{'objective': 'Write a weather report for SF today'}

In [13]:
OBJECTIVE = "What are the current news about crypto clarity act in the USA?"

In [14]:
baby_agi({"objective": OBJECTIVE})


*****TASK LIST*****

4: Create a detailed weather report for San Francisco for the next three days.
5: Generate a summary of the weather conditions in San Francisco for the upcoming week.
6: Provide a comparison of today's weather in San Francisco with the average for this time of year.
7: Analyze the weather trends in San Francisco for the past week.
8: Identify potential weather-related impacts on local events in San Francisco today.
9: Compile a list of local events happening in San Francisco today and their weather suitability.
10: Suggest clothing recommendations based on today's weather in San Francisco.
11: List outdoor activities suitable for today's weather in San Francisco.
12: List safety tips for outdoor activities in sunny weather in San Francisco.
13: Suggest hydration tips for staying cool in today's weather in San Francisco.
14: Draft a brief overview of the air quality in San Francisco today.
15: Research and report on the historical weather patterns for San Francisco

{'objective': 'What are the current news about crypto clarity act in the USA?'}